# 00 · Workspace Readiness for SLM SFT

Bare‑minimum health check for an Azure ML workspace and its compute before starting supervised fine‑tuning work in this repository.

**What this notebook verifies**

1. **Python interpreter** and required package versions (`torch`, `transformers ≥ 4.43`, `peft`, `trl`, `accelerate`, `bitsandbytes`, `datasets`, `sentencepiece`, `azure-ai-ml`, `mlflow`, plus optional `unsloth`).
2. **GPU** availability, CUDA version, VRAM, and bf16 support (target: T4‑class SKU such as `Standard_NC4as_T4_v3`, `NC6s_v3`, `NC8ads_T4_v3`).
3. **AML workspace** connectivity and presence of at least one GPU compute target.
4. **Hugging Face Hub** outbound reachability (downloads a tiny random Llama).
5. **Sample QLoRA SFT loop** (2 steps) so the whole stack compiles + trains without touching real data.
6. **Sample inference** call on the fine‑tuned tiny model.
7. A final **PASS/FAIL summary** table.

Run top‑to‑bottom on the same compute you plan to use for `03_azureml_fine_tuning.ipynb`.

## 0. Preflight - install repo dependencies (once per kernel)

This notebook assumes `requirements.txt` and `requirements-azureml.txt` are already installed in the active kernel. Uncomment the cell below on first run, then restart the kernel if any package was newly installed.


In [ ]:
# Uncomment on first run; restart the kernel afterwards if new packages were installed.
# %pip install --quiet -r ../requirements.txt -r ../requirements-azureml.txt


## 1. Environment: interpreter and packages

In [1]:
from __future__ import annotations

import importlib.metadata
import platform
import sys
from dataclasses import dataclass

RESULTS: list[tuple[str, bool, str]] = []

def record(name: str, ok: bool, detail: str = "") -> None:
    RESULTS.append((name, ok, detail))
    print(f"{'[OK]' if ok else '[FAIL]'} {name}" + (f" — {detail}" if detail else ""))

# Python version — repo pins 3.12.10 in environments/train-conda.yml.
py_version = platform.python_version()
record("python", py_version.startswith("3.12"), f"got {py_version}, expected 3.12.x")

REQUIRED = {
    "torch":            (">=", "2.4"),
    "transformers":     (">=", "4.43"),
    "peft":             (">=", "0.11"),
    "accelerate":       (">=", "0.33"),
    "trl":              (">=", "0.9"),
    "bitsandbytes":     (">=", "0.43"),
    "datasets":         (">=", "2.19"),
    "sentencepiece":    (">=", "0.2"),
    "azure-ai-ml":      (">=", "1.20"),
    "azure-identity":   (">=", "1.15"),
    "mlflow":           (">=", "2.14"),
    "huggingface_hub":  (">=", "0.24"),
    "safetensors":      (">=", "0.4"),
}
OPTIONAL = {
    "unsloth":         (">=", "2024.8"),
    "azureml-mlflow":  (">=", "1.55"),
    "flash-attn":      (">=", "2.5"),
}

def _tuple(v: str) -> tuple[int, ...]:
    parts = []
    for chunk in v.split("+")[0].split("."):
        head = "".join(c for c in chunk if c.isdigit())
        parts.append(int(head) if head else 0)
    return tuple(parts)

def check(pkg: str, op: str, wanted: str, required: bool) -> None:
    try:
        installed = importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        record(f"pkg:{pkg}", not required, "missing" + ("" if required else " (optional)"))
        return
    ok = _tuple(installed) >= _tuple(wanted) if op == ">=" else _tuple(installed) == _tuple(wanted)
    record(f"pkg:{pkg}", ok or not required, f"{installed} ({op}{wanted})" + ("" if required else " (optional)"))

for pkg, (op, ver) in REQUIRED.items():
    check(pkg, op, ver, required=True)
for pkg, (op, ver) in OPTIONAL.items():
    check(pkg, op, ver, required=False)

[OK] python — got 3.12.10, expected 3.12.x
[OK] pkg:torch — 2.11.0 (>=2.4)
[OK] pkg:transformers — 5.7.0 (>=4.43)
[OK] pkg:peft — 0.20.0 (>=0.11)
[OK] pkg:accelerate — 1.14.0 (>=0.33)
[FAIL] pkg:trl — missing
[FAIL] pkg:bitsandbytes — missing
[OK] pkg:datasets — 2.19.1 (>=2.19)
[FAIL] pkg:sentencepiece — missing
[OK] pkg:azure-ai-ml — 1.34.1 (>=1.20)
[OK] pkg:azure-identity — 1.17.0 (>=1.15)
[OK] pkg:mlflow — 2.22.5 (>=2.14)
[OK] pkg:huggingface_hub — 1.27.0 (>=0.24)
[OK] pkg:safetensors — 0.7.0 (>=0.4)
[OK] pkg:unsloth — missing (optional)
[OK] pkg:azureml-mlflow — 1.62.0.post5 (>=1.55) (optional)
[OK] pkg:flash-attn — missing (optional)


## 2. GPU: CUDA, VRAM, precision support

Target SKUs for T4‑class training: `Standard_NC4as_T4_v3`, `Standard_NC6s_v3`, `Standard_NC8ads_T4_v3`. VRAM ≥ 15 GB and CUDA ≥ 11.8 are the practical minima for QLoRA on a 1–3B base.

In [2]:
import torch

cuda_ok = torch.cuda.is_available()
record("cuda:available", cuda_ok, torch.version.cuda or "no CUDA")
if cuda_ok:
    idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(idx)
    vram_gb = props.total_memory / (1024**3)
    record("gpu:device", True, f"{props.name} ({vram_gb:.1f} GiB, cc={props.major}.{props.minor})")
    record("gpu:vram>=15GiB", vram_gb >= 15.0, f"{vram_gb:.1f} GiB")
    record("gpu:bf16", torch.cuda.is_bf16_supported(), "bfloat16 supported" if torch.cuda.is_bf16_supported() else "will fall back to fp16")
else:
    record("gpu:device", False, "no CUDA device visible — SFT will be prohibitively slow")

[FAIL] cuda:available — no CUDA
[FAIL] gpu:device — no CUDA device visible — SFT will be prohibitively slow


## 3. AML workspace and compute targets

Uses `DefaultAzureCredential` — works with an Azure CLI session locally, and with the assigned managed identity on AML compute. Set `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, `AZUREML_WORKSPACE_NAME` in the environment (a `.env` at the repo root is loaded automatically).

In [3]:
import os, sys
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lib").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

try:
    from azure.ai.ml import MLClient
    from azure.identity import DefaultAzureCredential

    credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
    ml_client = MLClient(
        credential=credential,
        subscription_id=os.environ["AZURE_SUBSCRIPTION_ID"],
        resource_group_name=os.environ["AZURE_RESOURCE_GROUP"],
        workspace_name=os.environ["AZUREML_WORKSPACE_NAME"],
    )
    workspace = ml_client.workspaces.get(os.environ["AZUREML_WORKSPACE_NAME"])
    record("aml:workspace", True, workspace.name)

    computes = list(ml_client.compute.list())
    gpu_computes = [c for c in computes if str(getattr(c, "size", "")).lower().startswith(("standard_nc", "standard_nd"))]
    record("aml:compute_count", len(computes) > 0, f"{len(computes)} target(s) total")
    record(
        "aml:gpu_compute",
        len(gpu_computes) > 0,
        ", ".join(f"{c.name}={c.size}" for c in gpu_computes) or "no NC/ND SKU visible",
    )
except KeyError as exc:
    record("aml:workspace", False, f"missing env var {exc}")
except Exception as exc:
    record("aml:workspace", False, f"{type(exc).__name__}: {exc}")

[OK] aml:workspace — sriks-aml-ws
[OK] aml:compute_count — 3 target(s) total
[OK] aml:gpu_compute — gpu-cluster1=Standard_NC8as_T4_v3, NC16asT4V3=Standard_NC16as_T4_v3


## 4. Hugging Face Hub reachability

Downloads a tiny random Llama checkpoint (no gated auth, ~1 MB). Verifies outbound HTTPS, cache write access, and tokenizer + model parity.

In [4]:
TINY_MODEL_ID = "hf-internal-testing/tiny-random-LlamaForCausalLM"

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(TINY_MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(TINY_MODEL_ID, torch_dtype=torch.float32)
    if torch.cuda.is_available():
        model = model.to("cuda")
    record("hf:download", True, f"{TINY_MODEL_ID} ({sum(p.numel() for p in model.parameters())} params)")
except Exception as exc:
    record("hf:download", False, f"{type(exc).__name__}: {exc}")
    tokenizer = model = None

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

c:\code\raft-finetuning-slm\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vism\.cache\huggingface\hub\models--hf-internal-testing--tiny-random-LlamaForCausalLM. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The following generation flags are not valid and may be ignored: ['pad_token_id']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/21 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[OK] hf:download — hf-internal-testing/tiny-random-LlamaForCausalLM (1032272 params)


## 5. Sample QLoRA SFT loop (2 steps)

Trains a LoRA adapter on 8 synthetic examples for 2 steps. Loss just needs to be finite and decreasing on this toy task — the goal is to exercise `peft` + `trl` + `bitsandbytes` end‑to‑end, not to learn anything.

In [5]:
sft_ok = False
sft_detail = ""
try:
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model
    from transformers import TrainingArguments
    from trl import SFTConfig, SFTTrainer

    if tokenizer is None or model is None:
        raise RuntimeError("HF download stage failed; cannot run SFT")

    examples = [
        {"text": f"### Q: What is {a}+{b}?\n### A: {a + b}\n"}
        for a in range(2, 6) for b in range(1, 3)
    ]
    dataset = Dataset.from_list(examples)

    peft_config = LoraConfig(
        r=4, lora_alpha=8, lora_dropout=0.0, bias="none",
        target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM",
    )

    sft_args = SFTConfig(
        output_dir=str(PROJECT_ROOT / "output" / "readiness_sft"),
        per_device_train_batch_size=2,
        max_steps=2,
        learning_rate=1e-3,
        logging_steps=1,
        report_to=[],
        save_strategy="no",
        bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
        fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
        dataset_text_field="text",
        max_seq_length=64,
    )
    trainer = SFTTrainer(
        model=model, args=sft_args, train_dataset=dataset,
        tokenizer=tokenizer, peft_config=peft_config,
    )
    train_output = trainer.train()
    loss = float(train_output.training_loss)
    sft_ok = loss == loss and loss > 0  # non-NaN, non-zero
    sft_detail = f"final_loss={loss:.4f}, steps={sft_args.max_steps}"
    model = trainer.model
except Exception as exc:
    sft_detail = f"{type(exc).__name__}: {exc}"
record("sft:trl_qlora_2_steps", sft_ok, sft_detail)

[FAIL] sft:trl_qlora_2_steps — ModuleNotFoundError: No module named 'trl'


## 6. Sample inference

Runs a single greedy generation against the just‑trained tiny model. Failure here typically indicates a device‑map or dtype mismatch that would also break the real fine‑tune.

In [6]:
gen_ok = False
gen_detail = ""
try:
    if model is None or tokenizer is None:
        raise RuntimeError("Model unavailable")
    prompt = "### Q: What is 3+4?\n### A:"
    inputs = tokenizer(prompt, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs, max_new_tokens=8, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated = tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    gen_ok = True
    gen_detail = repr(generated)[:80]
except Exception as exc:
    gen_detail = f"{type(exc).__name__}: {exc}"
record("inference:generate_8_tokens", gen_ok, gen_detail)

[OK] inference:generate_8_tokens — 'logging Société заняめ диphysovis разли'


## 7. Summary

In [7]:
import pandas as pd

summary = pd.DataFrame(RESULTS, columns=["check", "passed", "detail"])
print(summary.to_string(index=False))

required_prefixes = ("python", "pkg:torch", "pkg:transformers", "pkg:peft", "pkg:trl", "pkg:accelerate",
                     "pkg:bitsandbytes", "pkg:azure-ai-ml", "gpu:device", "aml:workspace", "hf:download",
                     "sft:trl_qlora_2_steps", "inference:generate_8_tokens")
required = summary[summary["check"].isin(required_prefixes) | summary["check"].str.startswith(required_prefixes)]
failed = required[~required["passed"]]
if len(failed):
    raise AssertionError(f"{len(failed)} required check(s) failed:\n{failed.to_string(index=False)}")
print(f"\nAll {len(required)} required checks passed. Workspace is ready for SFT.")

                      check  passed                                                              detail
                     python    True                                        got 3.12.10, expected 3.12.x
                  pkg:torch    True                                                      2.11.0 (>=2.4)
           pkg:transformers    True                                                      5.7.0 (>=4.43)
                   pkg:peft    True                                                     0.20.0 (>=0.11)
             pkg:accelerate    True                                                     1.14.0 (>=0.33)
                    pkg:trl   False                                                             missing
           pkg:bitsandbytes   False                                                             missing
               pkg:datasets    True                                                     2.19.1 (>=2.19)
          pkg:sentencepiece   False                             

AssertionError: 4 required check(s) failed:
                check  passed                                                  detail
              pkg:trl   False                                                 missing
     pkg:bitsandbytes   False                                                 missing
           gpu:device   False no CUDA device visible — SFT will be prohibitively slow
sft:trl_qlora_2_steps   False              ModuleNotFoundError: No module named 'trl'